## OOP design of a Fitness Tracker

### 1. What are the attributes and methods of a fitness tracker?

In [3]:
from dateutil import parser
#It allows you to parse strings into Python datetime objects — automatically recognizing many date/time formats without you having to specify one manually
from helpers import gpsDistance

class Workout(object):
    """A class to keep track of workouts"""

    # Class variable to compute calories burned from workout time
    cal_per_hr = 200 # class variable representing calories per hour
    
    def __init__(self, start, end, calories=None):
        """Creates a workout class;  start and end are strings representing
        the start and end time (e.g., "1/1/2021 1:23 PM");  
        calories is an optional float specifying the calories burned 
        in the workout"""
        # note use of dateutil.parser to convert strings to datetime objects
        self.start = parser.parse(start)  
        self.end = parser.parse(end)
        self.icon = '😓'
        self.kind = 'Workout'
        self.calories = calories

    def get_calories(self):
        """Return the total calories burned in the workout"""
        if (self.calories == None):
            # calc the calories based on the length of the workout and cal_per_hr
            return Workout.cal_per_hr * (self.end - self.start).total_seconds() / 3600.0
        else:
            return self.calories

    def get_duration(self):
        """Return the duration of the workout, as a datetime.interval object"""
        return self.end-self.start

    def get_start(self):
        """Return the start time of the workout, as a datetime.datetime object"""
        return self.start

    def get_end(self):
        """Return the end time of the workout, as a datetime.datetime object"""
        return self.end

    def set_calories(self, calories):
        """Set the calories of the workout to calories"""
        self.calories = calories

    def set_start(self, start):
        """Set the start time of the workout to the supplied start string"""
        self.start = parser.parse(start)

    def set_end(self, end):
        """Set the start time of the workout to the supplied start string"""
        self.end = parser.parse(end)

    def get_kind(self):
        """Return the kind of the workout as a string"""
        return self.kind

    def __eq__(self, other):
        """Returns true if this workout is equal to another workout, false o.w."""
        # the \ breaks up the line
        return type(self) == type(other) and \
                self.start == other.start and \
                self.end == other.end and \
                self.kind == other.kind and \
                self.get_calories() == other.get_calories()

    def __str__(self):
        """Return a text-based graphical depiction of the workout"""
        width = 16
        retstr =  f"|{'–'*width}|\n"
        retstr += f"|{' ' *width}|\n"
        retstr += f"| {self.icon}{' '*(width-3)}|\n"  #assume width of icon is 2 chars - len('🏃🏽‍♀️');  doesn't do what you'd epxect
        retstr += f"| {self.kind}{' '*(width-len(self.kind)-1)}|\n"
        retstr += f"|{' ' *width}|\n"
        duration_str = str(self.get_duration())
        retstr += f"| {duration_str}{' '*(width-len(duration_str)-1)}|\n"
        cal_str = f"{round(self.get_calories(),1)}"
        retstr += f"| {cal_str} Calories {' '*(width-len(cal_str)-11)}|\n"

        retstr += f"|{' ' *width}|\n"
        retstr +=  f"|{'_'*width}|\n"

        return retstr


**Note:** It’s better to use getters and setters to access data attributes as this can prevent bugs due to changes in implementation

In [4]:
# TEST: Try get_calories() with and without calories being set
#        (Information Hiding Example)
# =============================================================================
# # Create a workout with calories set
# # Does not calculate cal number based on time diff, uses parameter directly
workout1 = Workout('9/30/2021 1:35 PM','9/30/2021 1:57 PM',400)
print(workout1.get_calories())
print(workout1.__dict__.keys()) # show the attribute names
print(workout1.__dict__.values()) # show the attribute values

# # Create a workout without calories set
# # Default is 200 cal per hr, calculates cal number based on time diff (22 mins)
workout2 = Workout('9/30/2021 1:35 PM','9/30/2021 1:57 PM')
print(workout2.get_calories())

400
dict_keys(['start', 'end', 'icon', 'kind', 'calories'])
dict_values([datetime.datetime(2021, 9, 30, 13, 35), datetime.datetime(2021, 9, 30, 13, 57), '😓', 'Workout', 400])
73.33333333333333


### Exercise 1
- Create one Workout object saved as variable `w_one`, from Jan 1 2021 at 3:30 PM until 4 PM. You want to estimate the calories from this workout. Print the number of calories for w_one.
- Create another Workout object saved as w_two, from Jan 1 2021 at 3:35 PM until 4 PM. You know you burned 300 calories for this workout. Print the number of calories for w_two. 

In [5]:
w_one = Workout('1/1/2021 3:30 PM', '1/1/2021 4:00 PM')
print(w_one.get_calories())

w_two = Workout('1/1/2021 3:35 PM', '1/1/2021 4:00 PM', 300)
print(w_two.get_calories())

100.0
300


### 2. Define subclasses

In [6]:
class RunWorkout(Workout):
    
    # new class variable
    cals_per_km = 100
    
    def __init__(self, start, end, elev=0, calories=None, route_gps_points=None):
        """Create a new instance of a running workout, where start and
        end are strings representing the start and end time of the workout,
        and elev is the total elevaation gain in the workout in feet,
        calories is an optional number representing the calories 
        burned in the run, and route_gps_points is an optional array 
        of (lat,lon) pairs representing the route of the run"""
        super().__init__(start,end,calories)
        self.icon = '🏃🏽‍'
        self.kind = 'Running'
        self.elev = elev
        self.route_gps_points = route_gps_points

    def get_elev(self):
        """Return the elevation gain of the workout in feet"""
        return self.elev

    def set_elev(self, e):
        """Sets the elevation gain of the workout in feet, to e"""
        self.elev = e

    def get_calories(self):
        """Return the total calories consumed by the workout, derived
        using 1) the GPS points if supplied, 2) calories, if supplied,
        or 3) an estimate of the calories based on the duration"""
        if (self.route_gps_points != None):
            dist = 0
            lastP = self.route_gps_points[0]
            for p in self.route_gps_points[1:]:
                dist += gpsDistance(lastP,p)
                lastP = p
            return dist * RunWorkout.cals_per_km
        else:
            return super().get_calories()

    def __eq__(self,other):
        """Returns true if this workout is equal to another workout, false o.w."""
        return super().__eq__(other) and self.elev == other.elev

In [7]:
points = [(42.3601,-71.0589),(42.3370,-71.2092)] # (latitude,longitude) pairs
run1 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:57 PM', 100, route_gps_points=points)
print(f'Cals with route points: {run1.get_calories()}')

run2 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:57 PM', 100) # 100 is elevation
print(f'Cals with super impl: {run2.get_calories()}')

Cals with route points: 1261.983237744608
Cals with super impl: 473.3333333333333


In [8]:
w1 = Workout('9/30/2021 1:35 PM','9/30/2021 2:05 PM', 500)
w2 = Workout('9/30/2021 1:35 PM','9/30/2021 2:05 PM') # cal are 200 by default
w3 = Workout('9/30/2021 1:35 PM','9/30/2021 2:05 PM', 100)

rw1 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:05 PM', 100)
rw2 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:05 PM', 200)
rw3 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:05 PM', 100)

print(w1 == w2)  # False since only length of workout is the same
print(w1 == w3)  # False since only length of workout is the same
print(w2 == w3)  # True since the length and calories are equal
print(w1 == rw1)  # False since the types of w1 and rw1 are not the same 
print(rw1 == rw2) # False since the elevation is different
print(rw1 == rw3) # True since the types, workout length, and elev is the same

False
False
True
False
False
True


In [9]:
class SwimWorkout(Workout):
    """Subclass of workout to representing swimming"""
    
    # redefine class variable cal_per_hr
    cal_per_hr = 400
    
    def __init__(self, start, end, pace, calories=None):
        """Create a new instance of a swimming workout, where start and
        end are strings representing the start and end time of the workout,
        and pace is the pace of the workout in min/100yd, and calories
        is an optional parameter specifying the calories burned in the workout
        """
        super().__init__(start,end,calories)
        self.icon = '🏊‍'
        self.kind = 'Swimming'
        self.pace = pace
    def get_pace(self):
        """Return the pace of the workout"""
        return self.pace
    def get_calories(self):
        """Return the total calories burned in the swim workout
           using the SwimWorkout cal_per_hr class variable"""
        if (self.calories == None):
            # calc the calories based on the length of the workout and cal_per_hr
            return SwimWorkout.cal_per_hr * (self.end - self.start).total_seconds() / 3600.0
        else:
            return self.calories

In [10]:
w = Workout('9/30/2021 1:35 PM','9/30/2021 1:57 PM') # uses 200 cal_per_hr
r = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:57 PM', 100) # uses 200 cal_per_hr
sw = SwimWorkout('9/30/2021 1:35 PM','9/30/2021 1:57 PM', 100) # uses 400 cal_per_hr

print(w)
print(r)
print(sw)

|––––––––––––––––|
|                |
| 😓             |
| Workout        |
|                |
| 0:22:00        |
| 73.3 Calories  |
|                |
|________________|

|––––––––––––––––|
|                |
| 🏃🏽‍             |
| Running        |
|                |
| 2:22:00        |
| 473.3 Calories |
|                |
|________________|

|––––––––––––––––|
|                |
| 🏊‍             |
| Swimming       |
|                |
| 0:22:00        |
| 146.7 Calories |
|                |
|________________|



### Exercise 2
Write a function to caculate the total calories for a list of workouts

In [11]:
def total_calories(workouts):
    total = 0
    for workout in workouts:
        total += workout.get_calories()
    return total

# test with four workouts
w1 = Workout('9/30/2021 1:35 PM','9/30/2021 2:05 PM') # 30 min workout
w2 = Workout('9/30/2021 4:35 PM','9/30/2021 5:05 PM') # 30 min workout
rw1 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:35 PM', 100) # 2 hr workout
rw2 = RunWorkout('9/30/2021 1:35 PM','9/30/2021 3:35 PM', 200) # 2 hr workout

print(total_calories([w1,w2,rw1,rw2])) 

1000.0
